<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-07-mcp-and-cloud-run/lesson-7.1-fastmcp/notebooks/GCP_Capstone_7.1_FastMCP.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 7.1 Build the Lane's MCP Server — Four Tools, One Identity, Streamable HTTP
**Netsetos GenAI Engineering — GCP Capstone** · Module 7 · rebuilt on the live lane, 8 September 2026

The UI is how a person reaches DocuMind. This lesson builds how an *agent* reaches it - Claude Desktop, Cursor, another team's ADK app, lesson 7.3's agent: an MCP server that is a fourth service of the kit, with the kit's identity rules and the kit's one `retrieve()`.

The file you write here **is** the file the kit ships (`deploy/services/mcp/server.py`). `make check` holds the two identical, the way 12.2's notebook owns the API.


## Setup — the kit, the lane, and who you are


In [ ]:
!pip install -q fastmcp==3.4.7 uvicorn==0.52.4 google-genai==2.22.0 google-cloud-firestore==2.30.0 google-auth==2.57.1 requests==2.34.2
# fastmcp 3, not 4: lesson 7.3's ADK client needs the mcp 1.x protocol library (google-adk[mcp]
# declares mcp<2) and fastmcp 4 pulls mcp>=2. The server (this lesson), the smoke test and the
# agent all pin the same side of that line. Re-check the pair when you bump either.

from google.colab import auth
auth.authenticate_user()             # Application Default Credentials - and gcloud - for this session

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project `make up` deployed the lane into (4.8)
REGION     = "us-central1"           # where the services run (deploy/Makefile: REGION)
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is what this server imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import json, os, subprocess, sys, time
import google.auth
from google.auth import impersonated_credentials
from google.auth.transport.requests import AuthorizedSession, Request

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                  # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
RAG_API_URL = f"https://documind-api-{NUMBER}.{REGION}.run.app"
MEMBER_SA   = f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com"    # on acme, zeta AND globex (make roster)
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"  # may invoke the API; on no roster

# The server reads its world from the environment, exactly as it will on Cloud Run (7.2). Two
# differences here: SELF_URL is localhost (the audience every token to THIS server must carry),
# and DOCUMIND_IMPERSONATE_SA mints the API token as a roster member, because a notebook has no
# metadata server to be anyone with.
os.environ.update({"GOOGLE_CLOUD_PROJECT": PROJECT_ID, "DOCUMIND_PROFILE": "gcp",
                   "RAG_API_URL": RAG_API_URL, "SELF_URL": "http://localhost:8000",
                   "DOCUMIND_IMPERSONATE_SA": MEMBER_SA, "FASTMCP_STATELESS_HTTP": "true"})
print("API:", RAG_API_URL, "| kit:", KIT)


## Cell 1: Who is calling, for which tenant — before any tool exists
Every request to the lane answers two questions, in this order, with code that already exists. The UI answers them for a person; this server answers them for an agent, with the same two modules. A tenant is never a tool argument the model fills in unchecked, and never a header anyone can set.


In [ ]:
from shared import documind_tools, iap, tenancy

# Two questions every request to the lane must answer, in this order, and the code that answers
# them is the API's own (shared/iap.py, shared/tenancy.py) - this server does not get a second copy.
#   WHO is calling?   iap.identity(headers, bearer_audience=SELF_URL): the bearer token, verified
#                     for THIS service's URL, must carry a verified email. Nothing else is trusted.
#   FOR WHICH TENANT? tenancy: the roster at tenants/{tenant}/members/{email}. A caller on one
#                     roster needs no argument; a caller on several NAMES one and the roster must agree.
for tenant in ("acme", "zeta", "globex"):
    print(f"{MEMBER_SA.split('@')[0]:16} on {tenant:6}: {tenancy.is_member(MEMBER_SA, tenant)}"
          f"   {OUTSIDER_SA.split('@')[0]:16} on {tenant:6}: {tenancy.is_member(OUTSIDER_SA, tenant)}")
print("tenant_for(member) :", tenancy.tenant_for(MEMBER_SA), "  (the FIRST roster - a member of several must name one)")
print("tenant_for(outsider):", tenancy.tenant_for(OUTSIDER_SA))


## Cell 2: The one retrieve, from a notebook
`shared/documind_tools.retrieve` is the single retrieval entry point every brain in Modules 6 to 8 calls. It reaches the API with an ID token for the API's URL. On Cloud Run that token is the service's own; in a notebook there is no metadata server, so `DOCUMIND_IMPERSONATE_SA` mints it as a roster member instead - the same thing `make smoke` does with gcloud.


In [ ]:
# The ONE retrieve() - shared/documind_tools.py - called from a notebook, as a roster member.
# With DOCUMIND_IMPERSONATE_SA set, its ID token is minted by impersonation (the same thing
# `gcloud auth print-identity-token --impersonate-service-account` does for `make smoke`), for
# the API's URL, with the email included. Without that hook it asks the metadata server, which
# is what it will do on Cloud Run.
out = documind_tools.retrieve("After how many years of continuous service does gratuity become payable?",
                              "acme", top_k=5, brain="mcp")
print("answerable:", out.get("answerable"), "| confidence:", out.get("confidence"),
      "| citations:", len(out.get("citations") or []), "| error:", out.get("error"))
print((out.get("answer") or "")[:300])
for c in out.get("citations") or []:
    print("  -", c["source_uri"].rsplit("/", 1)[-1], "p.", c.get("page"), "|", (c.get("quote") or "")[:80])


## Cell 3: The server file — four tools, written once
Four tools, all of them the lane's operations: `retrieve` (the one implementation), `list_documents` (the ingest worker's claims, 12.5), `corpus_stats` (chunk counts by type and kind), `calculate_processing_cost` (pure; the chaining demo in 7.3). Read `_caller()` and `_tenant_for()` first: that is where the two questions of Cell 1 are answered on every call.


In [ ]:
# The server, in full. The four tools are the lane's own operations; nothing here owns an index.
# THIS TEXT IS THE KIT'S FILE: deploy/services/mcp/server.py is extracted from this cell
# (deploy/extract_documind.py), and `make check` fails the moment the two differ.
SERVER_PY = '''
"""DocuMind MCP server - the lane's tool surface for agents you did not build (lesson 7.1).

The UI is how a person reaches the lane; this is how an agent reaches it: Claude Desktop,
Cursor, another team's ADK app, lesson 7.3's agent. It is built the way the UI was built,
and for the same reasons:

  - a Cloud Run service with its own identity (documind-mcp-sa, terraform/sa.tf) that sits
    on the tenant rosters beside the UI's account;
  - the caller is verified by the same code the API uses (shared/iap.py, the bearer leg: an
    ID token minted for THIS service's URL, carrying a verified email);
  - the tenant comes from that identity and the roster (shared/tenancy.py) - never from a
    tool argument the model filled in, and never from a header anyone can set. A named
    tenant is accepted only after the roster confirms the caller is on it;
  - retrieval is the ONE implementation (shared/documind_tools.retrieve), which calls the
    API as this service's account. This file owns no index and must not grow one: an MCP
    server that starts caching documents is a second source of truth, and the day the two
    disagree is the day you stop trusting both.

Four tools, all of them the lane's own operations:

    retrieve                   grounded passages and the API's answer, in the citations contract
    list_documents             what is in the caller's corpus - the ingest worker's claims (12.5)
    corpus_stats               chunks and documents by type and by kind
    calculate_processing_cost  the six-key estimate; the tool-chaining demo (7.3)

Two lanes, one file (6.4's switch). DOCUMIND_PROFILE=gcp is the deployment above. With
DOCUMIND_PROFILE=local there is no token and no roster: the caller is LOCAL_USER, the tenant
LOCAL_TENANT, and retrieve() reads the Chroma directory shared/profile.py opens - the same
stand-in the chat service uses, and the Rs 0 lane of 13.2.

Transport: streamable HTTP at /mcp, STATELESS - every request carries its own identity, so an
instance keeps no session memory and Cloud Run can scale it freely. GET /health is for the
platform. Run it locally with

    PYTHONPATH=deploy SELF_URL=http://localhost:8000 RAG_API_URL=https://documind-api-NUMBER.us-central1.run.app \\\\
    DOCUMIND_IMPERSONATE_SA=documind-ui-sa@PROJECT.iam.gserviceaccount.com \\\\
    uvicorn services.mcp.server:app --port 8000

and deployed through commands/lesson-7.2.sh (`make build deploy-services`).
"""
from __future__ import annotations

import hashlib
import json
import logging
import os
import sys
from pathlib import Path

from fastmcp import FastMCP
from fastmcp.exceptions import ToolError
from fastmcp.server.dependencies import get_http_request
from starlette.requests import Request
from starlette.responses import JSONResponse

# The image copies deploy/shared beside this file (Dockerfile). A checkout runs it with
# PYTHONPATH=deploy, or as a script, in which case deploy/ is two directories up.
try:
    from shared import documind_tools, iap, tenancy
except ImportError:  # pragma: no cover - the script layout
    sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
    from shared import documind_tools, iap, tenancy

log = logging.getLogger("documind.mcp")
logging.basicConfig(level=logging.INFO, format="%(message)s")

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
SELF_URL = os.environ.get("SELF_URL", "")            # the audience every bearer token must carry
AUDIT_BUCKET = os.environ.get("AUDIT_BUCKET", "")    # set -> shared/audit_log rows; unset -> the log line only
DOC_TYPES = ("policy", "contract", "invoice", "report", "statute", "guidance", "form", "research_paper")

mcp = FastMCP(
    "DocuMind",
    instructions=(
        "DocuMind answers questions about a tenant's documents with citations. Use `retrieve` "
        "for any question about the documents; use `list_documents` to see what the corpus "
        "holds; `corpus_stats` for counts; `calculate_processing_cost` to price a set of pages. "
        "Never state a figure that a citation does not carry."
    ),
)


# ------------------------------------------------------------------ who is calling, for whom
def _caller() -> dict:
    """The identity behind this request: the bearer token's verified email, or the local stand-in."""
    if documind_tools.PROFILE == "local":
        return {"email": os.environ.get("LOCAL_USER", "dev@documind.local"), "via": "local"}
    if not SELF_URL:
        raise ToolError("SELF_URL is not configured on this server, so it cannot verify who is calling")
    try:
        headers = get_http_request().headers
    except Exception as e:  # noqa: BLE001 - an in-process client, no HTTP request to read
        raise ToolError(f"no HTTP request to read an identity from ({type(e).__name__})") from None
    try:
        return iap.identity(headers, bearer_audience=SELF_URL)
    except iap.IapError as e:
        # 401 in spirit: the token is missing, for another audience, or carries no email.
        raise ToolError(f"not authenticated: {e}") from None


def _tenant_for(caller: dict, named: str | None) -> str:
    """The tenant this call is for - from the roster, never from the argument alone.

    A caller on one roster needs no argument. A caller on several (an operator, or a
    notebook minting as documind-ui-sa) names one, and the roster must agree; naming a
    tenant you are not on is the 403 the API would give, raised here before the API is called.
    """
    if documind_tools.PROFILE == "local":
        return named or os.environ.get("LOCAL_TENANT", "acme")
    email = caller["email"]
    if named:
        if not tenancy.is_member(email, named):
            raise ToolError(f"{email} is not on tenant {named!r}'s roster")
        return named
    tenant = tenancy.tenant_for(email)
    if not tenant:
        raise ToolError(f"{email} is on no tenant's roster - an operator adds members with `make roster`")
    return tenant


def _audit(caller: dict, tenant: str, tool: str, meta: dict) -> None:
    """One line per call, always; one bucket row too when the audit bucket is configured."""
    log.info(json.dumps({"event": "mcp_call", "tool": tool, "tenant": tenant,
                         "caller": caller.get("email"), "via": caller.get("via"), **meta}))
    if AUDIT_BUCKET:
        try:
            from shared import audit_log
            audit_log.emit("query.submit",
                           actor={"email": caller.get("email"), "via": caller.get("via"), "tenant_id": tenant},
                           target={"surface": "mcp", "tool": tool, "tenant_id": tenant}, meta=meta)
        except Exception as e:  # noqa: BLE001 - the answer must not depend on the audit bucket
            log.warning('{"event":"mcp_audit_failed","error":"%s"}', type(e).__name__)


def _db():
    from google.cloud import firestore
    return firestore.Client(project=PROJECT or None)


# ------------------------------------------------------------------ the four tools
@mcp.tool(name="retrieve")
def retrieve(query: str, doc_type: str = "all", top_k: int = 5, tenant: str | None = None) -> dict:
    """Retrieve grounded passages from DocuMind's corpus, with the lane's own cited answer.

    Args:
        query: The question, in natural language.
        doc_type: policy, contract, invoice, report, statute, guidance, form, research_paper, or all.
        top_k: How many passages to return (1-20).
        tenant: Only if you belong to several tenants - which one. Checked against the roster.
    """
    if not query.strip():
        # A caller bug, not a runtime condition: the caller is the one who can fix it.
        raise ToolError("query cannot be empty")
    if doc_type not in ("all", "") and doc_type not in DOC_TYPES:
        raise ToolError(f"doc_type must be one of {DOC_TYPES} or all, not {doc_type!r}")
    caller = _caller()
    tenant_id = _tenant_for(caller, tenant)
    out = documind_tools.retrieve(query, tenant_id, top_k=max(1, min(int(top_k), 20)),
                                  doc_type=None if doc_type in ("all", "") else doc_type, brain="mcp")
    _audit(caller, tenant_id, "retrieve",
           {"query_sha": hashlib.sha256(query.encode("utf-8")).hexdigest()[:16],
            "answerable": out.get("answerable"), "citations": len(out.get("citations") or []),
            "error": out.get("error")})
    return out                       # the citations contract, or {"error": ...} as DATA the model can read


@mcp.tool
def list_documents(status: str = "indexed", tenant: str | None = None) -> dict:
    """What is in the caller's corpus: one row per uploaded document, from the ingest worker's claims.

    Args:
        status: indexed, processing, failed, or all.
        tenant: Only if you belong to several tenants - which one.
    """
    if status not in ("indexed", "processing", "failed", "all"):
        raise ToolError("status must be indexed, processing, failed or all")
    caller = _caller()
    tenant_id = _tenant_for(caller, tenant)
    if documind_tools.PROFILE == "local":
        docs = _local_documents(tenant_id)
    else:
        docs = []
        prefix = f"{tenant_id}_"                                   # documents/{tenant}_{sha256} (12.5)
        for snap in _db().collection("documents").stream():
            if not snap.id.startswith(prefix):
                continue
            d = snap.to_dict() or {}
            if status != "all" and d.get("status") != status:
                continue
            when = d.get("indexed_at") or d.get("failed_at") or d.get("claimed_at")
            docs.append({"file": (d.get("gcs_uri") or "").rsplit("/", 1)[-1], "status": d.get("status"),
                         "chunks": d.get("chunks"), "at": when.isoformat() if hasattr(when, "isoformat") else when,
                         "error": d.get("error")})
        docs.sort(key=lambda x: x["file"])
    _audit(caller, tenant_id, "list_documents", {"status": status, "documents": len(docs)})
    return {"tenant": tenant_id, "status": status, "documents": docs}


@mcp.tool
def corpus_stats(tenant: str | None = None) -> dict:
    """Chunks and documents in the caller's corpus, by document type and by kind (text, figure, table, segment).

    Args:
        tenant: Only if you belong to several tenants - which one.
    """
    caller = _caller()
    tenant_id = _tenant_for(caller, tenant)
    by_type: dict[str, int] = {}
    by_kind: dict[str, int] = {}
    sources: set[str] = set()
    for meta in _chunk_metas(tenant_id):
        by_type[meta.get("doc_type") or "unknown"] = by_type.get(meta.get("doc_type") or "unknown", 0) + 1
        by_kind[meta.get("kind") or "text"] = by_kind.get(meta.get("kind") or "text", 0) + 1
        if meta.get("source_uri"):
            sources.add(meta["source_uri"])
    out = {"tenant": tenant_id, "chunks": sum(by_type.values()), "documents": len(sources),
           "by_doc_type": dict(sorted(by_type.items(), key=lambda kv: -kv[1])), "by_kind": by_kind}
    _audit(caller, tenant_id, "corpus_stats", {"chunks": out["chunks"], "documents": out["documents"]})
    return out


@mcp.tool
def calculate_processing_cost(total_pages: int, num_documents: int = 1, processing_type: str = "standard") -> dict:
    """Estimate document processing cost in USD and INR.

    Args:
        total_pages: Total page count across all documents.
        num_documents: How many documents those pages are spread across.
        processing_type: Service tier - standard, priority, or bulk.
    """
    try:
        return documind_tools.calculate_processing_cost(int(total_pages), int(num_documents), processing_type)
    except ValueError as e:
        raise ToolError(str(e)) from None


# ------------------------------------------------------------------ the two stores, read only
def _chunk_metas(tenant_id: str) -> list:
    if documind_tools.PROFILE == "local":
        from shared.profile import build_store
        got = build_store().get(where={"tenant_id": tenant_id}, include=["metadatas"])
        return list(got.get("metadatas") or [])
    from google.cloud.firestore_v1.base_query import FieldFilter
    q = (_db().collection("chunks").where(filter=FieldFilter("tenant_id", "==", tenant_id))
         .select(["doc_type", "kind", "source_uri"]))               # never the 768 floats
    return [snap.to_dict() or {} for snap in q.stream()]


def _local_documents(tenant_id: str) -> list:
    """The local lane has no claims table: a document is a distinct source_uri in the store."""
    files: dict[str, int] = {}
    for meta in _chunk_metas(tenant_id):
        name = (meta.get("source_uri") or "").rsplit("/", 1)[-1]
        if name:
            files[name] = files.get(name, 0) + 1
    return [{"file": f, "status": "indexed", "chunks": n, "at": None, "error": None} for f, n in sorted(files.items())]


# ------------------------------------------------------------------ the app
@mcp.custom_route("/health", methods=["GET"])
async def health(request: Request) -> JSONResponse:
    return JSONResponse({"status": "ok", "profile": documind_tools.PROFILE, "self_url": SELF_URL or None})


app = mcp.http_app(path="/mcp", stateless_http=True)

if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=int(os.environ.get("PORT", "8080")))
'''

os.makedirs(f"{KIT}/deploy/services/mcp", exist_ok=True)
os.chdir(f"{KIT}/deploy/services/mcp")
with open('server.py', 'w') as f:
    f.write(SERVER_PY)
diff = subprocess.run(["git", "-C", KIT, "diff", "--stat", "--", "deploy/services/mcp/server.py"],
                      capture_output=True, text=True).stdout.strip()
print(f"server.py: {len(SERVER_PY.splitlines())} lines |", "identical to the kit's committed file" if not diff else diff)


### The other two files: requirements and the Dockerfile
The image builds from `deploy/`, like the chat service, so `shared/` lands beside the server. Nothing is copied into this directory by hand.


In [ ]:
# The other two files the service ships. Same rule: written here, extracted to the kit, held by `make check`.
REQUIREMENTS_TXT = '''
# DocuMind MCP server (lesson 7.1). The image builds from deploy/ so shared/ lands beside it.
# fastmcp 3, not 4: lesson 7.3's ADK client needs the mcp 1.x protocol library (google-adk[mcp]
# declares mcp<2), and fastmcp 4 pulls mcp>=2. Server and client pin the same side of that line.
fastmcp==3.4.7
uvicorn==0.52.4
# needed by shared/: the per-call ID token (documind_tools), the bearer-token verifier (iap),
# the roster and the ingest claims (tenancy, this server), the audit bucket (audit_log).
requests==2.34.2
google-auth==2.57.1
google-cloud-firestore==2.30.0
google-cloud-storage==3.13.1
'''

DOCKERFILE = '''
FROM python:3.12-slim
RUN useradd --create-home --shell /bin/bash --uid 10001 app
WORKDIR /app
# BUILD CONTEXT IS deploy/, not deploy/services/mcp/ - the server imports the shared layer
# (the one retrieve(), the bearer-token verifier, the roster), so shared/ lands beside it:
#     gcloud builds submit . --config=cloudbuild.yaml \\
#       --substitutions=_IMAGE=...,_DOCKERFILE=services/mcp/Dockerfile      # run from deploy/
COPY services/mcp/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY --chown=app:app shared/ ./shared/
COPY --chown=app:app services/mcp/ ./
USER app
ENV PYTHONUNBUFFERED=1 FASTMCP_STATELESS_HTTP=true
# Cloud Run sets PORT. Stateless streamable HTTP: no session to pin, so no --session-affinity.
CMD ["sh", "-c", "uvicorn server:app --host 0.0.0.0 --port ${PORT:-8080}"]
'''

with open('requirements.txt', 'w') as f:
    f.write(REQUIREMENTS_TXT)
with open('Dockerfile', 'w') as f:
    f.write(DOCKERFILE)
print(sorted(os.listdir(".")))


## Cell 4: Run it here, against the live lane
Streamable HTTP at `/mcp`, **stateless**: every request carries its own identity, so an instance keeps no session memory and Cloud Run can scale it freely (7.2). `/health` is for the platform.


In [ ]:
# Run it here, in a subprocess, the way uvicorn runs it in the container - against the LIVE lane.
# PYTHONPATH=deploy is what `from shared import ...` needs on a checkout; the image copies shared/ instead.
env = {**os.environ, "PYTHONPATH": f"{KIT}/deploy"}
SERVER = subprocess.Popen([sys.executable, "-m", "uvicorn", "services.mcp.server:app", "--port", "8000",
                           "--log-level", "warning"], cwd=f"{KIT}/deploy", env=env)
import requests
for _ in range(40):
    time.sleep(0.5)
    try:
        r = requests.get("http://localhost:8000/health", timeout=2)
        if r.ok:
            break
    except requests.RequestException:
        pass
print("health:", r.status_code, r.json())


## Cell 5: Call it as a roster member
The client sends one credential: an ID token minted for *this* server's URL. `documind-ui-sa` sits on three rosters, so it names the tenant, and the server checks the roster before the API is called. Every response is the citations contract 3.2 defined and the API returns - passed through, never reshaped.


In [ ]:
from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

SCOPE = ["https://www.googleapis.com/auth/cloud-platform"]

def id_token_as(service_account: str, audience: str) -> str:
    """A Google ID token minted AS a service account, for one audience, with the email - 4.8's Cell 1.
    The AUDIENCE is this server's SELF_URL: it verifies the token was minted for it and nothing else."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account, target_scopes=SCOPE)
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token

MCP_LOCAL = "http://localhost:8000"

async def call(token, name, args=None):
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    async with Client(StreamableHttpTransport(f"{MCP_LOCAL}/mcp", headers=headers)) as c:
        if name == "tools":
            return [t.name for t in await c.list_tools()]
        r = await c.call_tool(name, args or {})
        return r.data if getattr(r, "data", None) is not None else r

MEMBER = id_token_as(MEMBER_SA, MCP_LOCAL)
print("tools:", await call(MEMBER, "tools"))

# ui-sa sits on three rosters, so it NAMES the tenant; the server checks the roster before the API is called.
out = await call(MEMBER, "retrieve", {"query": "After how many years of continuous service does gratuity become payable?", "tenant": "acme"})
print("\nretrieve ->", out.get("answerable"), out.get("confidence"), len(out.get("citations") or []), "citations")
print(" ", (out.get("answer") or "")[:200])
docs = await call(MEMBER, "list_documents", {"status": "all", "tenant": "acme"})
print("\nlist_documents ->", len(docs["documents"]), "documents;", {d["status"] for d in docs["documents"]})
for d in docs["documents"][:5]:
    print("  ", d["file"], d["status"], d["chunks"])
print("\ncorpus_stats ->", await call(MEMBER, "corpus_stats", {"tenant": "acme"}))
print("\ncost ->", await call(MEMBER, "calculate_processing_cost", {"total_pages": 119, "num_documents": 5, "processing_type": "priority"}))


## Cell 6: Three refusals, three reasons
Not authenticated, authenticated but on no roster, on a roster but naming another tenant, and a caller's bug. Each is a `ToolError` with a sentence the client can read. None of them reaches the API.


In [ ]:
# Three refusals, three different reasons - and every one is a ToolError the client can read, never a 500.
async def expect_refusal(label, token, args):
    try:
        out = await call(token, "retrieve", args)
        print(f"{label:34} UNEXPECTED answer: {str(out)[:90]}")
    except Exception as e:
        print(f"{label:34} refused: {str(e)[:110]}")

await expect_refusal("no token", None, {"query": "gratuity?"})                                   # not authenticated
await expect_refusal("the outsider (on no roster)", id_token_as(OUTSIDER_SA, MCP_LOCAL), {"query": "gratuity?"})   # authenticated, not authorised
await expect_refusal("a member naming the wrong tenant", MEMBER, {"query": "gratuity?", "tenant": "nobody"})       # the roster disagrees
await expect_refusal("an empty query", MEMBER, {"query": "   "})                                                    # the caller's bug


### Errors that are data
A caller's mistake is an exception. A backend failure is a value - `answerable=False` with an error string - so the model can explain instead of inventing.


In [ ]:
# A backend failure is DATA, not an exception: the model can say retrieval is down instead of the
# turn dying. Point the one retrieve() at nowhere for one call and read the shape that comes back.
saved = documind_tools.RAG_API_URL
documind_tools.RAG_API_URL = "https://documind-api-nowhere.invalid"
try:
    print(documind_tools.retrieve("gratuity?", "acme", brain="mcp"))
finally:
    documind_tools.RAG_API_URL = saved
# A tool that INVENTS plausible documents when its backend is down is worse than one that fails:
# the model quotes the invention. answerable=False with an error string is the honest shape.


## Cell 7: The same server on the Rs 0 lane
`DOCUMIND_PROFILE=local` is 6.4's switch, inherited: no token, no roster, no cloud. The caller is `LOCAL_USER`, the tenant `LOCAL_TENANT`, and `retrieve()` reads the Chroma directory the kit seeds from the same corpus. The agent code that calls the server does not change; that is the contract the switch exists to keep.


In [ ]:
# The same server on the Rs 0 lane (6.4's switch): no token, no roster, no cloud. The caller is
# LOCAL_USER, the tenant LOCAL_TENANT, and retrieve() reads the Chroma directory the kit seeds from
# the SAME corpus (deploy/shared/local_corpus.py). Optional here - it installs Chroma.
RUN_LOCAL_LANE = False
if RUN_LOCAL_LANE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chromadb==1.5.9", "langchain-chroma==1.1.0", "langchain-core==1.6.2"], check=True)
    local_env = {**env, "DOCUMIND_PROFILE": "local", "LOCAL_TENANT": "acme", "DOCUMIND_CHROMA_DIR": "/content/documind_chroma"}
    subprocess.run([sys.executable, "-m", "shared.local_corpus", "acme"], cwd=f"{KIT}/deploy", env=local_env, check=True)
    LOCAL = subprocess.Popen([sys.executable, "-m", "uvicorn", "services.mcp.server:app", "--port", "8001", "--log-level", "warning"],
                             cwd=f"{KIT}/deploy", env=local_env)
    time.sleep(6)
    async with Client(StreamableHttpTransport("http://localhost:8001/mcp")) as c:
        r = await c.call_tool("retrieve", {"query": "After how many years of continuous service does gratuity become payable?"})
        out = r.data if getattr(r, "data", None) is not None else r
        print("local lane ->", out.get("answerable"), len(out.get("citations") or []), "citations (lexical, no cloud)")
    LOCAL.terminate()


## Cell 8: Stop the server


In [ ]:
SERVER.terminate()
print("server stopped. What you wrote is deploy/services/mcp/server.py; 7.2 deploys it as documind-mcp.")


## ✅ Lesson 7.1 complete

- ✅ An MCP server that is the lane's agent surface, built the way the UI was built
- ✅ Identity from the bearer token, the tenant from the roster - the API's own code, not a copy
- ✅ Four tools that are lane operations; retrieval through the one `retrieve()`
- ✅ The file you wrote is the kit's file (`make check`)
- ✅ Refusals as `ToolError`s, backend failures as data, and the Rs 0 lane on the same switch

**Next: 7.2 — deploy it as `documind-mcp`, as its own account, on the rosters; then the three calls that prove the boundary is identity, not the network.**
